# 02 · 传统 AEC 交互教程
目标：逐块看 **MDF 自适应收敛** 与 **DTD 三态状态机**，理解双讲检测为什么这么难。
配套：`exp/06_trad_aec/`、`docs/adr/0001-dtd-mic-vs-yecho-coherence.md`。

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '..'); sys.path.insert(0, '../exp/06_trad_aec')
from pipeline import TradAEC
rng = np.random.default_rng(0); sr = 16000; n = sr*4

# 造一个简单房间冲激响应（指数衰减）
L = 1024; rir = rng.standard_normal(L) * np.exp(-np.arange(L)/200); rir /= np.abs(rir).max()
far = rng.standard_normal(n) * 0.5                     # 远端
echo = np.convolve(far, rir)[:n]                        # 回声
near = np.zeros(n)
near[sr*2:sr*3] = rng.standard_normal(sr) * 0.8         # 2~3s 双讲（近端介入）
mic = echo + near + 0.01*rng.standard_normal(n)
print('FEST段=0~2s, 双讲段=2~3s, 远端单讲回声占主导')

## 1. 跑全链路，看 ERLE 随时间收敛（✏️ 改 mu 步长）
观察：FEST 段 ERLE 快速爬升（滤波器收敛）；双讲段 DTD 冻结防止近端被吃掉。

In [ ]:
aec = TradAEC(method='mdf', mu=0.7, use_dtd=True, use_res=False)  # ✏️ 改 mu / use_dtd
res = aec.process(mic, far)
B = aec.block; nb = len(mic)//B
erle_t = [10*np.log10((mic[i*B:(i+1)*B]**2).mean()/((res.error[i*B:(i+1)*B]**2).mean()+1e-12))
          for i in range(nb)]
tt = np.arange(nb)*B/sr
fig, ax = plt.subplots(figsize=(11,3.5))
ax.plot(tt, erle_t, label='block ERLE (dB)')
ax.axvspan(2,3,color='r',alpha=0.15,label='double-talk')
ax.axhline(20,ls='--',c='g',alpha=0.5,label='DoD 20dB')
ax.set_xlabel('s'); ax.set_ylabel('ERLE dB'); ax.legend(); ax.grid(alpha=0.3)
plt.show(); print(f'双讲块占比 dt_frac={res.dt_frac:.2f}, 时延估计={res.delay_ms:.1f}ms')

## 2. 对照实验：关掉 DTD 会发生什么？
把上面 `use_dtd=True` 改成 `False` 重跑——双讲段近端会被自适应滤波器'学进回声路径'，
近端被吃掉（ERLE 在双讲段异常飙高，但那是近端被误消，不是真回声抑制）。
这就是 ADR 0001 里 DTD 存在的意义。

## 3. 面试自检
1. 写出 NLMS 的更新方程，mu 的稳定范围是多少？
2. 双讲为什么必须冻结自适应？不冻结会怎样？
3. 为什么用 mic-vs-y_echo 相干而不是 mic-vs-far 时域相关？（段内泄漏）
4. ERLE 应该只在什么段统计？（FEST，跳过收敛瞬态）